## 1. Knowledge Base Design

The KB is a set of **ground facts** (edges, distances, road attributes, town attributes)
plus **rules** that let the agent infer useful conclusions (reachability, feasibility,
next action).

### Ontology (predicates)
| Predicate | Meaning |
|---|---|
| `Edge(u, v, km)` | directed highway segment from u to v of length km |
| `Road(name, u, v)` | highway name for the segment |
| `Surface(name, tarmac/gravel)` | road surface |
| `SpeedLimit(name, kph)` | legal speed limit |
| `FuelStop(town)` | town has a fuel station |
| `Town(town, region, population)` | town attributes |

### Rules (forward chaining)
1. `Reachable(a, b)` ← `Edge(a, b, _)`
2. `Reachable(a, c)` ← `Reachable(a, b) ∧ Edge(b, c, _)`
3. `Feasible(u, v, fuel)` ← `Edge(u, v, d) ∧ fuel ≥ d`
4. `NextStop(u, goal, fuel)` ← feasible successor on cheapest route
5. `NeedFuel(u, fuel)` ← `¬∃v FuelStop(v) ∧ …` (fuel below reserve)

In [6]:

# KNOWLEDGE BASE — ground facts
# Edges/distances come from PE3 Part 1.
# Road names, surface, speed limits & fuel stops are additional
# domain knowledge (Google Maps + field knowledge).


EDGES = [
    # (from, to, km, road, surface, speed_kph)
    ("Nairobi",  "Naivasha", 90,  "A104", "tarmac", 100),
    ("Nairobi",  "Narok",    142, "B3",   "tarmac", 100),
    ("Naivasha", "Nakuru",   69,  "A104", "tarmac", 100),
    ("Narok",    "Bomet",    82,  "B3",   "tarmac", 90),
    ("Nakuru",   "Kericho",  107, "B1",   "tarmac", 100),
    ("Nakuru",   "Eldoret",  154, "A104", "tarmac", 100),
    ("Bomet",    "Kericho",  77,  "B1",   "tarmac", 90),
    ("Kericho",  "Kisumu",   83,  "B1",   "tarmac", 100),
    ("Eldoret",  "Kisumu",   117, "A1",   "tarmac", 100),
]

TOWNS = {
    "Nairobi":  {"region": "Nairobi",         "fuel": True},
    "Naivasha": {"region": "Rift Valley",     "fuel": True},
    "Narok":    {"region": "Rift Valley",     "fuel": True},
    "Nakuru":   {"region": "Rift Valley",     "fuel": True},
    "Bomet":    {"region": "Rift Valley",     "fuel": True},
    "Kericho":  {"region": "Rift Valley",     "fuel": True},
    "Eldoret":  {"region": "Rift Valley",     "fuel": True},
    "Kisumu":   {"region": "Lake Victoria",   "fuel": True},
}

# Build graph + derived KB tables 
graph   = {t: {} for t in TOWNS}
road_info = {}                      # (u,v) -> dict
for u, v, km, road, surf, spd in EDGES:
    graph[u][v] = km
    road_info[(u, v)] = {"road": road, "surface": surf, "speed": spd}

print("KB loaded:", len(EDGES), "edges across", len(TOWNS), "towns")

KB loaded: 9 edges across 8 towns


In [7]:

#   - facts: Edge/4, Town/4, Road/5
#   - forward chaining: Reachable/2 (incremental)
#   - provenance: every fact remembers the rule that produced it
#   - queries: reachable, fuel-ok (with reason), cheapest route, fewest hops
import heapq
from collections import deque

#ground facts (from PE3 Part 1) 
EDGES = [
    ("Nairobi",  "Naivasha", 90,  "A104", "tarmac", 100),
    ("Nairobi",  "Narok",    142, "B3",   "tarmac", 100),
    ("Naivasha", "Nakuru",   69,  "A104", "tarmac", 100),
    ("Narok",    "Bomet",    82,  "B3",   "tarmac",  90),
    ("Nakuru",   "Kericho",  107, "B1",   "tarmac", 100),
    ("Nakuru",   "Eldoret",  154, "A104", "tarmac", 100),
    ("Bomet",    "Kericho",  77,  "B1",   "tarmac",  90),
    ("Kericho",  "Kisumu",   83,  "B1",   "tarmac", 100),
    ("Eldoret",  "Kisumu",   117, "A1",   "tarmac", 100),
]

TOWNS = {
    "Nairobi":  {"region": "Nairobi",       "fuel": True},
    "Naivasha": {"region": "Rift Valley",   "fuel": True},
    "Narok":    {"region": "Rift Valley",   "fuel": True},
    "Nakuru":   {"region": "Rift Valley",   "fuel": True},
    "Bomet":    {"region": "Rift Valley",   "fuel": True},
    "Kericho":  {"region": "Rift Valley",   "fuel": True},
    "Eldoret":  {"region": "Rift Valley",   "fuel": True},
    "Kisumu":   {"region": "Lake Victoria", "fuel": True},
}

# derived graph + per-edge attributes
graph     = {t: {} for t in TOWNS}
road_info = {}
for u, v, km, road, surf, spd in EDGES:
    graph[u][v] = km
    road_info[(u, v)] = {"road": road, "surface": surf, "speed": spd}

# inference engine 
class KnowledgeBase:
    def __init__(self, edges, towns, graph):
        self.edges, self.towns, self.graph = edges, towns, graph
        self.facts          = set()
        self.justifications = {}
        self._load_facts()
        self._forward_chain()

    # provenance-aware assertion
    def _assert(self, fact, reason):
        if fact not in self.facts:
            self.facts.add(fact)
            self.justifications[fact] = reason

    def _load_facts(self):
        for u, v, km, road, surf, spd in self.edges:
            self._assert(("Edge", u, v, km), "given: PE3 directed graph")
            self._assert(("Road", road, u, v), f"given: road {road} for {u}->{v}")
        for t, a in self.towns.items():
            self._assert(("Town", t, a["region"], a["fuel"]),
                         "given: town attributes")

    def _forward_chain(self):
        adj = {t: set() for t in self.towns}
        for u, v, *_ in self.edges:
            adj[u].add(v)
            self._assert(("Reachable", u, v),
                         f"Rule-R1 (base): Edge({u},{v})")
        for src in adj:
            seen, queue = set(), list(adj[src])
            while queue:
                nxt = queue.pop()
                if nxt in seen:
                    continue
                seen.add(nxt)
                queue.extend(adj[nxt] - seen)
            for dst in seen:
                self._assert(("Reachable", src, dst),
                             f"Rule-R2 (transitive): {src}↝{dst}")

    # queries 
    def why(self, fact):
        return self.justifications.get(fact, "not in KB")

    def ask_reachable(self, a, b):
        return ("Reachable", a, b) in self.facts

    def ask_fuel_ok(self, u, v, fuel_km):
        d = self.graph.get(u, {}).get(v)
        if d is None:
            return False, f"No Edge({u},{v}) in KB"
        ok = fuel_km >= d
        return ok, (f"Rule-R3: fuel {fuel_km} "
                    f"{'≥' if ok else '<'} leg {u}→{v} = {d} km")

    def has_fuel_stop(self, town):
        return self.towns.get(town, {}).get("fuel", False)

    def cheapest_route(self, start, goal):
        frontier, best = [(0, start, [start])], {start: 0}
        while frontier:
            cost, u, path = heapq.heappop(frontier)
            if u == goal:
                return path, cost
            for v, w in self.graph.get(u, {}).items():
                nc = cost + w
                if nc < best.get(v, float("inf")):
                    best[v] = nc
                    heapq.heappush(frontier, (nc, v, path + [v]))
        return None, float("inf")

    def shortest_hops(self, start, goal):
        if start == goal:
            return [start]
        q, parent = deque([start]), {start: None}
        while q:
            u = q.popleft()
            for v in self.graph.get(u, {}):
                if v not in parent:
                    parent[v] = u
                    if v == goal:
                        path, cur = [], goal
                        while cur is not None:
                            path.append(cur); cur = parent[cur]
                        return list(reversed(path))
                    q.append(v)
        return None

# instantiate + smoke tests
KB = KnowledgeBase(EDGES, TOWNS, graph)

print("KB facts count            :", len(KB.facts))
print("Reachable Nairobi→Kisumu :", KB.ask_reachable("Nairobi", "Kisumu"))
print("Why?                     :", KB.why(("Reachable", "Nairobi", "Kisumu")))
print("Cheapest route (km)      :", KB.cheapest_route("Nairobi", "Kisumu"))
print("Fewest hops (edges)      :", KB.shortest_hops("Nairobi", "Kisumu"))
print("Fuel check (Nairobi→Naivasha, 400 km):",
      KB.ask_fuel_ok("Nairobi", "Naivasha", 400))
print("Fuel check (Nakuru→Eldoret, 100 km)  :",
      KB.ask_fuel_ok("Nakuru", "Eldoret", 100))

KB facts count            : 47
Reachable Nairobi→Kisumu : True
Why?                     : Rule-R2 (transitive): Nairobi↝Kisumu
Cheapest route (km)      : (['Nairobi', 'Naivasha', 'Nakuru', 'Kericho', 'Kisumu'], 349)
Fewest hops (edges)      : ['Nairobi', 'Naivasha', 'Nakuru', 'Kericho', 'Kisumu']
Fuel check (Nairobi→Naivasha, 400 km): (True, 'Rule-R3: fuel 400 ≥ leg Nairobi→Naivasha = 90 km')
Fuel check (Nakuru→Eldoret, 100 km)  : (False, 'Rule-R3: fuel 100 < leg Nakuru→Eldoret = 154 km')


In [8]:

# KNOWLEDGE-BASED AGENT
#  Percept : (current_town, fuel_km, goal)
#  Act     : drive(to) | refuel() | report(msg) | stop


class TrailerAgent:
    RESERVE_KM = 30                        # safety fuel reserve

    def __init__(self, kb, start, goal, fuel_km):
        self.kb        = kb
        self.state     = start
        self.goal      = goal
        self.fuel      = fuel_km
        self.history   = []                # (action, town, fuel_after, note)
        self.finished  = False

    #percept → tell KB
    def tell_percept(self):
        # Persist live percepts as facts (demonstrates TELL)
        self.kb.facts.add(("At", self.state))
        self.kb.facts.add(("Fuel", self.state, self.fuel))

    # ASK KB → decide action 
    def think(self):
        # 1. If at goal → done.
        if self.state == self.goal:
            return ("stop", None, "Goal reached")

        # 2. Find cheapest onward route.
        path, cost = self.kb.cheapest_route(self.state, self.goal)
        if path is None:
            return ("report", None, "No route to goal")

        next_town = path[1]
        leg_km    = self.kb.graph[self.state][next_town]

        # 3. Fuel feasibility check (rule Feasible/3).
        if not self.kb.ask_fuel_ok(self.state, next_town, self.fuel - self.RESERVE_KM):
            if self.kb.has_fuel_stop(self.state):
                return ("refuel", None, f"Low fuel ({self.fuel} km) before {next_town}")
            return ("report", None,
                    f"Cannot reach {next_town}: fuel {self.fuel} km, leg {leg_km} km, no station")

        return ("drive", next_town, f"drive {leg_km} km on {road_info[(self.state, next_town)]['road']}")

    # act
    def act(self, action, target, note):
        if action == "drive":
            leg_km = self.kb.graph[self.state][target]
            self.fuel   -= leg_km
            self.state   = target
            self.history.append(("drive", target, self.fuel, note))
        elif action == "refuel":
            self.fuel   = 600                 # fill tank
            self.history.append(("refuel", self.state, self.fuel, note))
        elif action == "report":
            self.history.append(("report", self.state, self.fuel, note))
            self.finished = True
        elif action == "stop":
            self.history.append(("stop", self.state, self.fuel, note))
            self.finished = True

    # -------- agent loop ---------------------------------------
    def run(self, max_steps=50, verbose=True):
        step = 0
        while not self.finished and step < max_steps:
            step += 1
            self.tell_percept()
            action, target, note = self.think()
            if verbose:
                tgt = f" → {target}" if target else ""
                print(f"[{step:>2}] at {self.state:<9} fuel={self.fuel:>3} km | "
                      f"{action}{tgt}  ({note})")
            self.act(action, target, note)
        if verbose:
            print(f"\nFinished at {self.state}, fuel={self.fuel} km, {step} steps")
        return self.history


#  run the agent 
agent  = TrailerAgent(KB, start="Nairobi", goal="Kisumu", fuel_km=400)
hist   = agent.run()

[ 1] at Nairobi   fuel=400 km | drive → Naivasha  (drive 90 km on A104)
[ 2] at Naivasha  fuel=310 km | drive → Nakuru  (drive 69 km on A104)
[ 3] at Nakuru    fuel=241 km | drive → Kericho  (drive 107 km on B1)
[ 4] at Kericho   fuel=134 km | drive → Kisumu  (drive 83 km on B1)
[ 5] at Kisumu    fuel= 51 km | stop  (Goal reached)

Finished at Kisumu, fuel=51 km, 5 steps


In [9]:

# USER INTERACTION MODE  (safe, no mixed indentation)
#   Commands:
#     route <from> <to>
#     reachable <a> <b>
#     fuel <town> <km> <to>
#     info <town>
#     drive
#     help
#     quit


HELP_TEXT = (
    "route <a> <b>       cheapest route + cost\n"
    "reachable <a> <b>   yes/no\n"
    "fuel <town> <km> <to>  can I drive there with that fuel?\n"
    "info <town>         town attributes\n"
    "drive               run the trailer agent\n"
    "help                this message\n"
    "quit                exit\n"
)


def interactive(kb, agent):
    print("Trailer KB agent ready. Type 'help' for commands.\n")
    while True:
        try:
            line = input("trailer> ").strip()
        except EOFError:
            break
        if not line:
            continue

        parts = line.split()
        cmd = parts[0].lower()

        if cmd == "quit":
            break

        elif cmd == "help":
            print(HELP_TEXT)

        elif cmd == "route" and len(parts) == 3:
            path, cost = kb.cheapest_route(parts[1], parts[2])
            if path:
                print(" -> ".join(path), "| cost:", cost, "km")
            else:
                print("No route from", parts[1], "to", parts[2])

        elif cmd == "reachable" and len(parts) == 3:
            print(kb.ask_reachable(parts[1], parts[2]))

        elif cmd == "fuel" and len(parts) == 4:
            town, fuel_km, to = parts[1], int(parts[2]), parts[3]
            ok, reason = kb.ask_fuel_ok(town, to, fuel_km)
            print("OK" if ok else "INSUFFICIENT FUEL", "-", reason)

        elif cmd == "info" and len(parts) == 2:
            print(kb.towns.get(parts[1], "Unknown town"))

        elif cmd == "drive":
            agent.run()

        else:
            print("Unknown or malformed command. Type 'help'.")



In [10]:
# Optional enrichment — public weather API (no key required)
# Adds road-condition facts that the agent can consult.
import urllib.request, json

def fetch_weather(town, lat, lon):
    url = (f"https://api.open-meteo.com/v1/forecast?"
           f"latitude={lat}&longitude={lon}&current_weather=true")
    with urllib.request.urlopen(url, timeout=10) as r:
        return json.load(r)["current_weather"]

town_coords = {
    "Nairobi":  (-1.2864, 36.8172),
    "Naivasha": (-0.7167, 36.4310),
    "Nakuru":   (-0.3031, 36.0800),
    "Kericho":  (-0.3689, 35.2860),
    "Kisumu":   (-0.0917, 34.7680),
}

for town, (lat, lon) in town_coords.items():
    try:
        w = fetch_weather(town, lat, lon)
        KB.facts.add(("Weather", town, w["temperature"], w["windspeed"]))
        print(f"{town:<9} temp={w['temperature']}°C wind={w['windspeed']} km/h")
    except Exception as e:
        print(f"{town:<9} weather unavailable ({e})")

Nairobi   temp=19.0°C wind=8.9 km/h
Naivasha  temp=19.4°C wind=4.8 km/h
Nakuru    temp=19.8°C wind=14.5 km/h
Kericho   temp=17.3°C wind=3.3 km/h
Kisumu    temp=25.2°C wind=0.6 km/h
